# División (Split) del dataset unificado de perfiles completos y fracciones

Con el fin de preparar los datos para el entrenamiento de modelos, en este notebook se genera una metadata persistente con una partición estratificada.

El objetivo es que otros notebooks puedan reutilizar exactamente la misma división `train` / `val` / `test` sin recalcularla.

El resultado se guarda como `metadata.csv` con una nueva columna `split`.

`CLASES_A_DIVIDIR` selecciona las clases que se procesan. Sus imágenes se copian a `DATA_DESTINO` conservando las carpetas de clase y las subcarpetas originales. La división `train` / `val` / `test` queda registrada en la metadata, sin crear carpetas por split. Al repetir la ejecución se sobrescriben los archivos coincidentes; no se eliminan archivos de ejecuciones anteriores.

In [1]:
# Importaciones
from vasijas.archivos import copiar_carpetas_de_clases
from vasijas.config import load_config
from vasijas.metadata import cargar_metadata, guardar_metadata
from vasijas.paths import ProjectPaths, ruta_relativa
from vasijas.split_dataset import (
    ConfiguracionSplitDataset,
    copiar_imagenes_split,
    realizar_split_en_metadata_por_id_perfil_completo,
    describir_configuracion_split,
    advertir_imagenes_sin_metadata,
    mostrar_resumen_split,
    construir_resumen_split,
)

In [2]:
# Configuracion general
CONFIG = load_config()
PATHS = ProjectPaths.from_root(CONFIG.project_root)

## Clases a dividir


In [3]:
CLASES_A_DIVIDIR = [1, 2, 3, 5, 6, 7, 8, 9, 10, 11]
print(f"Clases a dividir: {CLASES_A_DIVIDIR}")

Clases a dividir: [1, 2, 3, 5, 6, 7, 8, 9, 10, 11]


## Origen y destino de datos

In [4]:
# Origen y destino de datos
DATA_ORIGEN = PATHS.DATA_UNIFICADO
DATA_DESTINO = PATHS.DATA_UNIFICADO_DIVIDIDO

print(f"Dataset de imágenes a dividir: {ruta_relativa(DATA_ORIGEN)}")
print(f"Destino de imágenes y archivo de metadata: {ruta_relativa(DATA_DESTINO)}")

Dataset de imágenes a dividir: data\11.unificado
Destino de imágenes y archivo de metadata: data\12.unificado_divididos_en_train_val_y_test


## Copia de imágenes

A fin seguir el criterio de colocar en una misma carpeta las imágenes a procesar y el archivo metadata se copian las imágenes a la carperta de destino.

In [5]:
# Copiar toda la estructura de DATA_CORREGIDOS a DATA_UBICADOS
archivos_copiados = copiar_carpetas_de_clases(
    DATA_ORIGEN, DATA_DESTINO, CLASES_A_DIVIDIR
)

print(f"{archivos_copiados} archivos copiados.")
print(f"Desde {ruta_relativa(DATA_ORIGEN)}\n" f"Hacia {ruta_relativa(DATA_DESTINO)}")

20349 archivos copiados.
Desde data\11.unificado
Hacia data\12.unificado_divididos_en_train_val_y_test


## Vista previa al procesamiento

In [6]:
METADATA_ORIGEN = cargar_metadata(DATA_ORIGEN)

print("Muestra metadata origen:")
METADATA_ORIGEN

Muestra metadata origen:


,Identificador,Tiene asa,Tendencia Formal,Borde,Cuello,Cuerpo,Base,Clase Ana,Diametro Borde,Diametro Base,Altura Total,Cronologia,Ubicacion,id_perfil_completo,borde_origen,porcentaje_avance
0,al_03a,No,II,1,B,2,2,11.0,182,54,53,ibero,Almedinilla,al_03a,completo,100.0
1,al_03b,No,II,1,B,2,2,11.0,178,57,46,ibero,Almedinilla,al_03b,completo,100.0
2,al_03c,No,II,1,B,2,2,11.0,185,56,49,ibero,Almedinilla,al_03c,completo,100.0
3,al_03d,No,II,1,B,2,2,11.0,160,41,45,ibero,Almedinilla,al_03d,completo,100.0
4,al_03e,No,II,1,B,2,2,11.0,164,54,44,ibero,Almedinilla,al_03e,completo,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20695,tu_96_8_tope_50_centrado,No,II,1,B,2,1,11.0,188,95,40,ibero,Tútugi,tu_96_8,tope,50.0
20696,tu_96_8_tope_60_centrado,No,II,1,B,2,1,11.0,188,95,40,ibero,Tútugi,tu_96_8,tope,60.0
20697,tu_96_8_tope_70_centrado,No,II,1,B,2,1,11.0,188,95,40,ibero,Tútugi,tu_96_8,tope,70.0
20698,tu_96_8_tope_80_centrado,No,II,1,B,2,1,11.0,188,95,40,ibero,Tútugi,tu_96_8,tope,80.0


## Configuración del split

In [7]:
# Parámetros del split
SEED = 42

# TODO MEJORAR, PASAR A UNA FUNCIÓN
config_split = ConfiguracionSplitDataset(semilla=SEED)

print(describir_configuracion_split(config_split))

Split configurado -> train: 70%, val: 15%, test: 15%, semilla: 42


## Construcción del split

Se usa `torchvision.datasets.ImageFolder` para obtener las imágenes de las clases seleccionadas con `CLASES_A_DIVIDIR`. Se dividen los valores únicos de `id_perfil_completo` de forma estratificada por clase: primero `train` contra el conjunto temporal, y después `val` contra `test`.

Cada perfil completo y todas sus fracciones reciben el mismo `split`. Las proporciones configuradas se aplican a perfiles únicos; las proporciones de imágenes pueden variar según la cantidad de fracciones de cada perfil. Todas las imágenes seleccionadas deben tener metadata con `id_perfil_completo` y cada perfil debe pertenecer a una sola clase.

In [8]:
resultado_split = realizar_split_en_metadata_por_id_perfil_completo(
    data_origen=DATA_ORIGEN,
    metadata_origen=METADATA_ORIGEN,
    configuracion_split=config_split,
    clases_a_dividir=CLASES_A_DIVIDIR,
)

df_split = resultado_split.registros_split

mostrar_resumen_split(resultado_split)

Clases procesadas: ['clase01', 'clase02', 'clase03', 'clase05', 'clase06', 'clase07', 'clase08', 'clase09', 'clase10', 'clase11']
Imágenes procesadas: 20349
Split -> train: 14231, val: 3059, test: 3059


In [9]:
# Tabla de archivos con split
df_split.head()

,Identificador,archivo,ruta_relativa,clase_carpeta,split
0,al_13b,al_13b.png,clase01/al_13b.png,clase01,train
1,al_13b_base_10_centrado,al_13b_base_10_centrado.png,clase01/al_13b_base_10_centrado.png,clase01,train
2,al_13b_base_20_centrado,al_13b_base_20_centrado.png,clase01/al_13b_base_20_centrado.png,clase01,train
3,al_13b_base_30_centrado,al_13b_base_30_centrado.png,clase01/al_13b_base_30_centrado.png,clase01,train
4,al_13b_base_40_centrado,al_13b_base_40_centrado.png,clase01/al_13b_base_40_centrado.png,clase01,train


## Unión con metadata

La metadata se limita a las imágenes procesadas y se agregan las columnas `archivo`, `ruta_relativa`, `clase_carpeta` y `split`.


In [10]:
df_metadata_split = resultado_split.metadata
advertir_imagenes_sin_metadata(resultado_split.ids_sin_metadata)
df_metadata_split.head()

,Identificador,Tiene asa,Tendencia Formal,Borde,Cuello,Cuerpo,Base,Clase Ana,Diametro Borde,Diametro Base,Altura Total,Cronologia,Ubicacion,id_perfil_completo,borde_origen,porcentaje_avance,archivo,ruta_relativa,clase_carpeta,split
0,al_03a,No,II,1,B,2,2,11.0,182,54,53,ibero,Almedinilla,al_03a,completo,100.0,al_03a.png,clase11/al_03a.png,clase11,val
1,al_03b,No,II,1,B,2,2,11.0,178,57,46,ibero,Almedinilla,al_03b,completo,100.0,al_03b.png,clase11/al_03b.png,clase11,test
2,al_03c,No,II,1,B,2,2,11.0,185,56,49,ibero,Almedinilla,al_03c,completo,100.0,al_03c.png,clase11/al_03c.png,clase11,val
3,al_03d,No,II,1,B,2,2,11.0,160,41,45,ibero,Almedinilla,al_03d,completo,100.0,al_03d.png,clase11/al_03d.png,clase11,train
4,al_03e,No,II,1,B,2,2,11.0,164,54,44,ibero,Almedinilla,al_03e,completo,100.0,al_03e.png,clase11/al_03e.png,clase11,val


In [11]:
# Control de distribución por split y clase
resumen_split = construir_resumen_split(df_metadata_split)
resumen_split

split,test,train,val,All
clase_carpeta,,,,
clase01,76,342,76,494
clase02,133,608,133,874
clase03,209,950,190,1349
clase05,152,741,171,1064
clase06,152,684,133,969
clase07,133,646,133,912
clase08,798,3724,798,5320
clase09,57,266,57,380
clase10,1007,4712,1026,6745


In [12]:
# Copia de imágenes seleccionadas y guardado de metadata
imagenes_copiadas = copiar_imagenes_split(
    data_origen=DATA_ORIGEN,
    data_destino=DATA_DESTINO,
    registros_split=df_split,
)

ruta_metadata_split = guardar_metadata(
    df=df_metadata_split,
    carpeta_destino=DATA_DESTINO,
)

print(f"Metadata con split guardada en: {ruta_relativa(ruta_metadata_split)}")
print(f"Registros guardados: {len(df_metadata_split)}")
print(f"Imágenes copiadas: {imagenes_copiadas}")

Metadata con split guardada en: data\12.unificado_divididos_en_train_val_y_test\metadata.csv
Registros guardados: 20349
Imágenes copiadas: 20349


## Uso desde otros notebooks

```python
import pandas as pd

from vasijas.config import load_config
from vasijas.paths import ProjectPaths

CONFIG = load_config()
PATHS = ProjectPaths.from_root(CONFIG.project_root)

metadata_split = pd.read_csv(PATHS.DATA_DIVIDIDO / "metadata.csv")
train_df = metadata_split[metadata_split["split"] == "train"]
val_df = metadata_split[metadata_split["split"] == "val"]
test_df = metadata_split[metadata_split["split"] == "test"]
```
